In [160]:
################ Import necessary libraries

%pip install pyblp
%pip install statsmodels

import pandas as pd
import numpy as np
import pyblp 
import statsmodels.formula.api as smf

pyblp.options.digits = 2
pyblp.options.verbose = False
pyblp.__version__

Note: you may need to restart the kernel to use updated packages.
  Using cached statsmodels-0.14.5-cp311-cp311-win_amd64.whl.metadata (9.8 kB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   -------------------------

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\Lenovo\\AppData\\Local\\Temp\\pip-unpack-13t33cwx\\statsmodels-0.14.5-cp311-cp311-win_amd64.whl'
Check the permissions.



'1.1.2'

In [174]:
################ Load data ready for nested logit
df = pd.read_csv('data_with_IV.csv')

# Generate nesting_id
df['nesting_ids'] = pd.factorize(df['fuel_type'])[0]

# Generate log of charging station stock
df['log_charging_IV'] = np.log(df['charging_stations_stock_lag']) 

# Create indicator for electric vehicles
df['is_electric'] = df['type'].apply(lambda x: 1 if x == '国产新能源乘用车' else 0)

# Make year an object variable
df['year'] = df['year'].astype(str)


In [ ]:
# Rename columns to match pyblp requirements
df.rename(columns={
    'product_id': 'product_ids',
    'market_id': 'market_ids',
    'weighted_Avg_Price': 'prices',
    'market_share': 'shares',
    'cost_shifter' : 'demand_instruments0',
    'sum_rival_mass' : 'demand_instruments1',
    'sum_rival_power' : 'demand_instruments2',
    'euclidean_mass' : 'demand_instruments3',
    'euclidean_power' : 'demand_instruments4'
    'local_mass' : 'demand_instruments5',
    'local_power' : 'demand_instruments6'
}, inplace=True)


In [180]:
df.columns

Index(['year', 'type', 'province', 'brand', 'model', 'fuel_type', 'mass',
       'power', 'sales', 'prices', 'market_size', 'shares', 'province_id',
       'brand_id', 'model_id', 'product_ids', 'market_ids',
       'charging_stations_stock', 'charging_stations_stock_lag',
       'sum_rival_mass', 'sum_rival_power', 'euclidean_mass',
       'euclidean_power', 'local_mass', 'local_power', 'steel_price_index',
       'demand_instruments0', 'nesting_ids', 'log_charging_IV', 'is_electric',
       'demand_instruments1'],
      dtype='object')

In [ ]:
################ Nested Logit Model with Charging

def solve_nl(df):
    groups = df.groupby(['market_ids', 'nesting_ids'])
    df['demand_instruments1'] = groups['shares'].transform(np.size)
    nl_formulation = pyblp.Formulation('0 + prices + is_electric*log_charging_IV')
    problem = pyblp.Problem(nl_formulation, df)
    return problem.solve(rho=0.7)

# Solve the nested logit model
results = solve_nl(df)

# Display results
print(results)

Problem Results Summary:
GMM   Objective    Projected    Reduced   Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Hessian   Shares   Condition Number  Condition Number 
----  ---------  -------------  --------  -------  ----------------  -----------------
 2    +3.1E+03     +0.0E+00     +0.0E+00     0         +3.2E+08          +1.9E+02     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective 
   Time      Converged   Iterations   Evaluations
-----------  ---------  ------------  -----------
 00:00:22       Yes          1             5     

Rho Estimates (Robust SEs in Parentheses):
All Groups
----------
 +0.0E+00 
(+1.4E-02)

Beta Estimates (Robust SEs in Parentheses):
  prices    is_electric*log_charging_IV
----------  ---------------------------
 +6.3E-03            -1.4E-01          
(+1.1E-03)          (+2.4E-03)         


In [ ]:
################ Nested Logit Model with Charging using more IVs

def solve_nl_2(df):
    groups = df.groupby(['market_ids', 'nesting_ids'])
    df['demand_instruments7'] = groups['shares'].transform(np.size)
    nl_formulation = pyblp.Formulation('0 + prices + is_electric*log_charging_IV')
    problem = pyblp.Problem(nl_formulation, df)
    return problem.solve(rho=0.7)

# Solve the nested logit model
results_2 = solve_nl_2(df)

# Display results
print(results_2)

Problem Results Summary:
GMM   Objective    Projected    Reduced   Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Hessian   Shares   Condition Number  Condition Number 
----  ---------  -------------  --------  -------  ----------------  -----------------
 2    +3.1E+03     +0.0E+00     +0.0E+00     0         +3.2E+08          +1.9E+02     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective 
   Time      Converged   Iterations   Evaluations
-----------  ---------  ------------  -----------
 00:00:22       Yes          1             5     

Rho Estimates (Robust SEs in Parentheses):
All Groups
----------
 +0.0E+00 
(+1.4E-02)

Beta Estimates (Robust SEs in Parentheses):
  prices    is_electric*log_charging_IV
----------  ---------------------------
 +6.3E-03            -1.4E-01          
(+1.1E-03)          (+2.4E-03)         


In [ ]:
# Ensure log(sales) and log(charging_stations_stock_lag) are available
df_1 = df[df['year'] == 2020].copy()
df_1['log_sales'] = np.log(df['sales'])

# Run OLS regression with province fixed effects
model = smf.ols('log_sales ~ prices + mass + power + log_charging_IV + C(province)', data=df_1).fit()

print(model.summary())

In [187]:
print(results_2)

Problem Results Summary:
GMM   Objective    Projected    Reduced   Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Hessian   Shares   Condition Number  Condition Number 
----  ---------  -------------  --------  -------  ----------------  -----------------
 2    +1.3E-21     +2.4E-08     +2.7E+04     0         +1.8E+13          +1.3E+05     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective 
   Time      Converged   Iterations   Evaluations
-----------  ---------  ------------  -----------
 00:02:25       No           4            31     

Rho Estimates (Robust SEs in Parentheses):
All Groups
----------
 +6.6E-01 
(+8.6E-03)

Beta Estimates (Robust SEs in Parentheses):
  prices    is_electric  log_charging_IV  is_electric*log_charging_IV
----------  -----------  ---------------  ---------------------------
 -8.1E-02    -1.2E+01       -5.4E-01               +1.1E+00          
(+1.2E-03)  (+1.0E-01)     (+6.1E-03)             (+1.1E-02)

In [ ]:
print(results_2.problem)

Dimensions:
 T     N     K1    MD    H 
---  -----  ----  ----  ---
155  79290   4     5     4 

Formulations:
     Column Indices:          0          1              2                      3             
--------------------------  ------  -----------  ---------------  ---------------------------
X1: Linear Characteristics  prices  is_electric  log_charging_IV  is_electric*log_charging_IV